# Experiment 9
## Crop Disease Classification Using Leaf Images

### Decision Tree, KNN and Naive Bayes

### Aim
To classify crop diseases using leaf images by extracting numerical
features and applying Decision Tree, KNN and Gaussian Naive Bayes
classification algorithms.

### Learning Objectives
1. Download a labeled crop-leaf image dataset from Kaggle.
2. Convert leaf images into numerical feature vectors.
3. Train Decision Tree, KNN and Gaussian Naive Bayes models.
4. Compare models using accuracy, precision, recall, F1-score
   and confusion matrices.
5. Predict the class of a new leaf image.

## 2. Dataset Description

The PlantVillage dataset is used for crop disease classification.

It contains images of healthy and diseased plant leaves arranged
in different class folders.

Each folder represents a particular plant disease or healthy class.

Dataset Source:
https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset

## 3. Import Libraries

The required Python libraries are imported for:
- Image processing
- Numerical operations
- Feature extraction
- Machine learning model training
- Model evaluation and visualization

In [9]:
import pathlib
import numpy as np
import pandas as pd
import kagglehub

from PIL import Image
from skimage.color import rgb2gray
from skimage.feature import hog

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt

## 4. Download Dataset

The KaggleHub library is used to download the PlantVillage dataset
directly from Kaggle.

The downloaded dataset path is stored in a variable for further use.

In [3]:
import pathlib

# Find previously downloaded PlantVillage dataset
download_dir = pathlib.Path.home() / ".cache" / "kagglehub" / "datasets"

print("Searching for dataset...")

matches = list(download_dir.rglob("*plantvillage*"))

for folder in matches[:10]:
    print(folder)

Searching for dataset...
C:\Users\ASUS\.cache\kagglehub\datasets\abdallahalidev\plantvillage-dataset
C:\Users\ASUS\.cache\kagglehub\datasets\abdallahalidev\plantvillage-dataset\versions\3\plantvillage dataset


In [4]:
import kagglehub

path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")

print("Dataset downloaded successfully!")
print("Dataset Path:", path)

Dataset downloaded successfully!
Dataset Path: C:\Users\ASUS\.cache\kagglehub\datasets\abdallahalidev\plantvillage-dataset\versions\3


## 5. Set the Dataset Path

The downloaded dataset is inspected to identify the folder
containing individual plant disease classes.

Images are resized to 64 × 64 pixels.

A maximum of 300 images per class is selected to reduce
processing time during the classroom demonstration.

In [5]:
DATA_DIR = pathlib.Path(path)

IMG_SIZE = (64, 64)

MAX_IMAGES_PER_CLASS = 300

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

class_dirs = []

for folder in DATA_DIR.rglob("*"):

    if folder.is_dir():

        image_files = [
            f for f in folder.iterdir()
            if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
        ]

        if len(image_files) > 0:
            class_dirs.append(folder)

class_dirs = sorted(class_dirs)

class_names = [f.name for f in class_dirs]

print("Number of classes:", len(class_names))
print("Class names:", class_names)

Number of classes: 114
Class names: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___healthy', 'Corn_(maize)___Northern_Leaf_Blight', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___healthy', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___healthy', 'Strawberry___Leaf_scorch', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___healthy', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Sp

## 6. Extract Numerical Features

Machine learning algorithms require numerical input.

Therefore, numerical features are extracted from each leaf image.

The following features are used:

1. RGB Pixel Features
2. Colour Statistics (Mean and Standard Deviation)
3. HOG (Histogram of Oriented Gradients) Texture Features

Each image is resized to 64 × 64 pixels and converted into
a numerical feature vector.

In [6]:
def extract_features(image_path):

    image = Image.open(image_path).convert("RGB").resize(IMG_SIZE)

    arr = np.asarray(image, dtype=np.float32) / 255.0

    pixel_features = arr.flatten()

    colour_features = np.concatenate([
        arr.mean(axis=(0,1)),
        arr.std(axis=(0,1))
    ])

    gray = rgb2gray(arr)

    hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2),
        block_norm="L2-Hys"
    )

    return np.concatenate([
        pixel_features,
        colour_features,
        hog_features
    ]).astype(np.float32)

## 7. Create the Feature Dataset

All leaf images are converted into numerical feature vectors.

X stores the extracted image features.

y stores the corresponding class labels.

The images are processed class-wise and the extracted features
are combined to create the final dataset.


In [10]:
X, y = [], []

for label, folder in enumerate(class_dirs):

    files = [
        f for f in folder.iterdir()
        if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
    ]

    files = sorted(files)

    if MAX_IMAGES_PER_CLASS is not None:
        files = files[:MAX_IMAGES_PER_CLASS]

    for img_path in files:

        try:
            X.append(extract_features(img_path))
            y.append(label)

        except Exception as error:
            print("Skipped:", img_path, error)

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("Feature matrix:", X.shape)
print("Labels:", np.bincount(y))
print("Total images:", len(y))
print("Feature vector length:", X.shape[1])

Feature matrix: (33681, 14058)
Labels: [300 300 275 300 300 300 300 300 300 300 300 300 300 300 300 300 300 300
 300 300 300 152 300 300 300 300 300 300 300 300 300 300 300 300 300 300
 300 300 300 300 275 300 300 300 300 300 300 300 300 300 300 300 300 300
 300 300 300 300 300 152 300 300 300 300 300 300 300 300 300 300 300 300
 300 300 300 300 300 300 275 300 300 300 300 300 300 300 300 300 300 300
 300 300 300 300 300 300 300 152 300 300 300 300 300 300 300 300 300 300
 300 300 300 300 300 300]
Total images: 33681
Feature vector length: 14058


## 8. Split the Dataset

The dataset is divided into training and testing sets.

80% of the data is used for training.

20% of the data is used for testing.

random_state=42 ensures reproducibility.

stratify=y maintains the class distribution in both sets.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (26944, 14058)
Testing data: (6737, 14058)


## 9. Create Machine Learning Models

Three classification algorithms are used:

1. Decision Tree Classifier
2. K-Nearest Neighbors (KNN)
3. Gaussian Naive Bayes

StandardScaler is used with KNN and Gaussian Naive Bayes
to standardize the numerical features.

In [12]:
models = {

    "Decision Tree": DecisionTreeClassifier(
        max_depth=15,
        random_state=42
    ),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(
            n_neighbors=5,
            metric="euclidean"
        ))
    ]),

    "Gaussian Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", GaussianNB())
    ])

}

## 10. Train and Compare Models

All three models are trained using the training dataset.

Predictions are made using the testing dataset.

The models are evaluated using:

1. Accuracy
2. Macro Precision
3. Macro Recall
4. Macro F1-score

The results are stored in a DataFrame for comparison.

In [ ]:
results = []
trained_models = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    trained_models[name] = model

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Macro Precision": precision_score(
            y_test, pred, average="macro", zero_division=0
        ),
        "Macro Recall": recall_score(
            y_test, pred, average="macro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_test, pred, average="macro", zero_division=0
        )
    })

results_df = pd.DataFrame(results)

print(results_df.round(4))

## 11. Classification Report and Confusion Matrix

The classification report displays precision, recall and F1-score
for each individual class.

The confusion matrix compares actual and predicted labels.

It helps identify correctly and incorrectly classified images.

In [ ]:
for name, model in trained_models.items():

    pred = model.predict(X_test)

    print("\n", name)

    print(classification_report(
        y_test,
        pred,
        target_names=class_names,
        zero_division=0
    ))

    cm = confusion_matrix(y_test, pred)

    plt.figure(figsize=(14, 10))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.title("Confusion Matrix - " + name)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 12. Predict a New Leaf Image

A new leaf image is converted into numerical features
using the same feature extraction function.

The trained models predict the class of the given leaf image.

The predicted class represents the plant disease or healthy category.

In [ ]:
new_path = pathlib.Path(r"C:\Users\ASUS\Downloads\my_leaf.jpg")

new_features = extract_features(new_path).reshape(1, -1)

for name, model in trained_models.items():

    prediction = model.predict(new_features)[0]

    print(name, "->", class_names[prediction])

## 13. Optional Experiments

Different hyperparameter values are tested to observe
their effect on model performance.

For KNN, different values of K are tested.

For Decision Tree, different maximum depths are tested.

Macro F1-score is used to compare the results.

In [ ]:
for k in [1, 3, 5, 7, 9]:

    knn = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=k))
    ])

    knn.fit(X_train, y_train)

    pred = knn.predict(X_test)

    print(k, f1_score(y_test, pred, average="macro"))

In [ ]:
for depth in [3, 5, 10, 15, None]:

    tree = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    tree.fit(X_train, y_train)

    pred = tree.predict(X_test)

    print(depth, f1_score(y_test, pred, average="macro"))

## 15. Conclusion

Crop disease classification was performed using leaf images
by extracting RGB pixel, colour and HOG texture features.

Decision Tree, KNN and Gaussian Naive Bayes classifiers
were trained and evaluated using accuracy, precision,
recall, F1-score and confusion matrices.

The performance of all three classifiers was compared.
The classifier with the highest macro F1-score was
considered the best-performing model.

The experiment demonstrated how machine learning algorithms
can be used for crop disease classification.

The experiment also highlighted the importance of feature
extraction, feature scaling and hyperparameter selection.

## References

1. Kaggle PlantVillage Dataset:
   https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset

2. Kaggle PlantVillage Collection:
   https://www.kaggle.com/plantvillage/datasets

3. Scikit-learn Documentation:
   https://scikit-learn.org/stable/